# Morning Briefing Guide

Your daily market intelligence digest. Loads the pre-computed briefing for any date,
renders each spiking topic with its narrative summary, source articles, and sector
sentiment cross-check.

**No API key required** — briefings are pre-generated by CI and stored in `data/briefings/`.

## Sections
1. [Configuration](#1-configuration)
2. [Load briefing](#2-load-briefing)
3. [Spike narratives](#3-spike-narratives)
4. [Sector cross-check](#4-sector-cross-check)
5. [Browse history](#5-browse-history)

---
## 1 — Configuration

In [ ]:
from datetime import date

DATE   = date.today().isoformat()   # "YYYY-MM-DD" — change to any past date
TOP_N  = 5                          # max spikes to display

---
## 2 — Load briefing

Reads the pre-computed `data/briefings/{DATE}.json` written by the daily CI workflow.
If today's file is missing, prints instructions — briefings are generated each weekday at ~13:00 UTC.

In [ ]:
import json
import sys
from pathlib import Path
from datetime import date as _date, datetime

PROJECT_ROOT = Path("..") if Path("../pipeline").exists() else Path(".")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRIEFINGS_DIR = PROJECT_ROOT / "data" / "briefings"

def _load_briefing(date_str: str) -> dict | None:
    path = BRIEFINGS_DIR / f"{date_str}.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

briefing = _load_briefing(DATE)

if briefing is None:
    # Find the most recent available briefing as a fallback.
    available = sorted(BRIEFINGS_DIR.glob("*.json"), reverse=True)
    if available:
        fallback_date = available[0].stem
        print(f"No briefing for {DATE}.")
        print(f"Loading most recent available: {fallback_date}")
        DATE = fallback_date
        briefing = _load_briefing(DATE)
    else:
        print(f"No briefings found in {BRIEFINGS_DIR}.")
        print("Briefings are generated by CI each weekday. Check back after 13:00 UTC.")
        briefing = None

if briefing:
    print(f"Loaded briefing for {briefing['date']}  |  {briefing['n_spikes']} spike(s)")

---
## 3 — Spike narratives

Each spike is a topic cluster whose article count significantly exceeded its
rolling baseline that day. The `spike_ratio` is current count ÷ baseline average
(e.g. 1.5x = 50% above normal). The narrative is a 3–4 sentence LLM summary
backed by real source articles.

In [ ]:
if not briefing:
    print("No briefing loaded — see section 2.")
else:
    spikes = briefing["spikes"][:TOP_N]
    sep = "-" * 72

    print(f"\n{'=' * 72}")
    print(f"  DAILY BRIEFING  {briefing['date']}  ({briefing['n_spikes']} spikes)")
    print(f"{'=' * 72}")

    if not spikes:
        print("  No spike signals for this date.")
    else:
        for i, spike in enumerate(spikes, 1):
            label = spike.get("label") or "(unlabeled)"
            ratio = spike.get("spike_ratio", 0)
            count = spike.get("article_count", 0)
            answer = spike.get("rag_answer", "")
            sources = spike.get("rag_sources", [])

            print(f"\n{sep}")
            print(f"  #{i}  {label}")
            print(f"       spike={ratio:.2f}x   articles={count}")
            print(sep)

            if answer:
                print(f"\n  NARRATIVE")
                # Word-wrap at 70 chars.
                words, line = answer.split(), "    "
                for w in words:
                    if len(line) + len(w) + 1 > 72:
                        print(line); line = "    " + w
                    else:
                        line += (" " if line != "    " else "") + w
                if line.strip():
                    print(line)

            if sources:
                print(f"\n  SOURCES ({len(sources)})")
                for s in sources:
                    print(f"    [{s.get('date','')}] {s.get('title','')[:68]}")

    print(f"\n{'=' * 72}")

---
## 4 — Sector cross-check

Each spike includes a sector signal: the 14-day sentiment trend for sectors
related to that topic. When the narrative and the sector signal *agree*
(topic bullish + sector improving), the signal is stronger.

In [ ]:
if briefing and briefing["spikes"]:
    spikes_with_sectors = [s for s in briefing["spikes"][:TOP_N] if s.get("sectors")]

    if not spikes_with_sectors:
        print("No sector cross-check data for this briefing date.")
    else:
        print(f"{'spike topic':<35} {'sector':<28} {'trend':<14} {'score':>6}")
        print("-" * 90)
        for spike in spikes_with_sectors:
            label = (spike.get("label") or "(unlabeled)")[:33]
            for sc in spike["sectors"]:
                direction = sc["trend_direction"]
                arrow = {"improving": "[+]", "deteriorating": "[-]", "stable": "[=]"}.get(direction, "[ ]")
                print(f"{label:<35} {sc['sector']:<28} {arrow} {direction:<10} {sc['mean_sentiment_score']:>+6.3f}")
            label = ""  # only print label once per spike

---
## 5 — Browse history

List all available briefing dates and quickly compare spike counts over time.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

all_files = sorted(BRIEFINGS_DIR.glob("*.json"), reverse=True)
print(f"Available briefings: {len(all_files)}  ({all_files[-1].stem} to {all_files[0].stem})\n")

# Load spike counts for history chart.
records = []
for f in all_files:
    try:
        b = json.loads(f.read_text(encoding="utf-8"))
        records.append({"date": f.stem, "n_spikes": b.get("n_spikes", 0)})
    except Exception:
        pass

df_hist = pd.DataFrame(records).sort_values("date")
df_hist["date"] = pd.to_datetime(df_hist["date"])

# Print last 10.
print(f"{'date':<12}  {'spikes':>6}  topics")
print("-" * 60)
for _, row in df_hist.tail(10).iloc[::-1].iterrows():
    b = json.loads((BRIEFINGS_DIR / f"{row['date'].date()}.json").read_text(encoding="utf-8"))
    labels = [s.get("label","")[:40] for s in b.get("spikes",[])[:3]]
    print(f"{str(row['date'].date()):<12}  {int(row['n_spikes']):>6}  {' / '.join(l for l in labels if l)}")

# Plot spike count over time.
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(df_hist["date"], df_hist["n_spikes"], color="#2c3e50", alpha=0.7)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45, ha="right")
ax.set_ylabel("Spike count")
ax.set_title("Daily briefing spike count — full history")
plt.tight_layout()
plt.show()